In [1]:
import numpy as np

from qiskit_optimization import QuadraticProgram
from qiskit.primitives import Sampler
from qiskit_optimization.algorithms import (
    MinimumEigenOptimizer,
    RecursiveMinimumEigenOptimizer,
    SolutionSample,
    OptimizationResultStatus,
)
from qiskit.visualization import plot_histogram
from qiskit_ibm_runtime import EstimatorV2 as Estimator
from qiskit.quantum_info import SparsePauliOp 
from qiskit.circuit import QuantumCircuit

In [2]:
Q, offset, T, N = np.load('../../out/diploid/qubo_data_both2.syncasm1001.cropped.gfa_normalisation_32.npy', allow_pickle=True)

In [3]:
mod = QuadraticProgram("QUBO test")
mod.binary_var_list(Q.shape[0])
mod.minimize(constant=offset, linear=None, quadratic=Q)
# print(mod.prettyprint())

In [4]:
op, offset = mod.to_ising()
print("offset: {}".format(offset))
# print("operator:")
# print(op)

offset: 1614.0


In [5]:
# create a QUBO
qubo = QuadraticProgram()
qubo.binary_var("x")
qubo.binary_var("y")
qubo.binary_var("z")
qubo.minimize(constant=2, linear=[1, -2, 3], quadratic={("x", "y"): 1, ("x", "z"): -1, ("y", "z"): 2})
print(qubo.prettyprint())

Problem name: 

Minimize
  x*y - x*z + 2*y*z + x - 2*y + 3*z + 2

Subject to
  No constraints

  Binary variables (3)
    x y z



In [6]:
small_op, small_offset = qubo.to_ising()
print("offset: {}".format(offset))
print("operator:")
print(small_op)

offset: 1614.0
operator:
SparsePauliOp(['IIZ', 'IZI', 'ZII', 'IZZ', 'ZIZ', 'ZZI'],
              coeffs=[-0.5 +0.j,  0.25+0.j, -1.75+0.j,  0.25+0.j, -0.25+0.j,  0.5 +0.j])


In [7]:
from qiskit.circuit.library import QAOAAnsatz

p = 5
circuit = QAOAAnsatz(cost_operator=op, reps=p, flatten=True)
# circuit.save_matrix_product_state(label='before_measure_mps')
circuit.measure_all()

# circuit.draw('mpl')

In [8]:
from qiskit_aer import AerSimulator
from qiskit_ibm_runtime.fake_provider import FakeKyiv

backend = FakeKyiv()
print(backend)
aer_simulator = AerSimulator.from_backend(backend, method="matrix_product_state", matrix_product_state_max_bond_dimension=5, mps_log_data="True")

In [9]:
ideal_aer = AerSimulator(method="matrix_product_state", matrix_product_state_max_bond_dimension=5, mps_log_data="True")

In [10]:
from qiskit.transpiler.preset_passmanagers import generate_preset_pass_manager
# Create pass manager for transpilation
pm = generate_preset_pass_manager(optimization_level=3,
                                    backend=aer_simulator)

candidate_circuit = pm.run(circuit)
# candidate_circuit.draw('mpl', fold=False, idle_wires=False)

In [11]:
# Create pass manager for transpilation
ideal_pm = generate_preset_pass_manager(optimization_level=3,
                                    backend=ideal_aer)

ideal_circuit = ideal_pm.run(circuit)
# candidate_circuit.draw('mpl', fold=False, idle_wires=False)

In [12]:
initial_gamma = np.pi
initial_beta = np.pi/2
init_params = [initial_gamma, initial_beta] * p

In [ ]:
parameter_binding = {ideal_circuit.parameters[i]: [init_params[i]] for i in range(len(init_params))}
result = ideal_aer.run(ideal_circuit, parameter_binds=[parameter_binding]).result()
data = result.data(0)
data

In [13]:
parameter_binding = {candidate_circuit.parameters[i]: [init_params[i]] for i in range(len(init_params))}
aer_simulator.options.shots = 1
result = aer_simulator.run(candidate_circuit, parameter_binds=[parameter_binding], num_shots=1).result()
data = result.data(0)
data

{'counts': {'0x41201cc251221': 1}}

In [59]:
objective_func_vals = [] # Global variable
def cost_func_estimator(params: list, ansatz: QuantumCircuit, hamiltonian: SparsePauliOp, estimator: Estimator):

    # transform the observable defined on virtual qubits to
    # an observable defined on all physical qubits
    isa_hamiltonian = hamiltonian.apply_layout(ansatz.layout)

    pub = (ansatz, isa_hamiltonian, params)
    job = estimator.run([pub])

    results = job.result()[0]
    data = results.data
    print(data)
    cost = results.data.evs
    

    objective_func_vals.append(cost)

    return cost

In [ ]:
estimator = Estimator(mode=ideal_aer)
estimator.options.default_shots = 1000
cost_func_estimator(
    init_params,
    ideal_circuit, op, estimator,
)

In [ ]:
from qiskit_ibm_runtime import Session, EstimatorV2 as Estimator
from scipy.optimize import minimize

with Session(backend=ideal_aer) as session:
    # If using qiskit-ibm-runtime<0.24.0, change `mode=` to `session=`
    estimator = Estimator(mode=session)
    estimator.options.default_shots = 1000

    # Set simple error suppression/mitigation options
    # estimator.options.dynamical_decoupling.enable = True
    # estimator.options.dynamical_decoupling.sequence_type = "XY4"
    # estimator.options.twirling.enable_gates = True
    # estimator.options.twirling.num_randomizations = "auto"

    result = minimize(
        cost_func_estimator,
        init_params,
        args=(ideal_circuit, op, estimator),
        method="COBYLA",
        tol=1e-2,
    )
    print(result)

p = 1
x: [ 3.063e+00  1.648e+00]

p = 5
x: [ 3.123e+00  1.592e+00  3.126e+00  1.566e+00  3.139e+00
            1.558e+00  3.149e+00  2.620e+00  4.138e+00  1.582e+00]



Pretty close to initialised values??
